<a href="https://colab.research.google.com/github/nayeaong/kcc/blob/main/SDE_Entropy_QuasarT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SDE + Entropy Quasar-T 실험
- 엔트로피 기반 SDE/NL 동적 분기
- 데이터셋: Quasar-T
- 모델: Qwen2.5-7B-Instruct


In [1]:
# ── STEP 1: 환경 설정 ──────────────────────────────
import os
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache/hub"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
print(f"캐시 경로: {os.environ['HF_HOME']}")


캐시 경로: /content/hf_cache


In [2]:
# ── STEP 2: 레포 클론 ──────────────────────────────
REPO_DIR = "/content/StateDelta_Entropy"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/LittleDinoC/StateDelta.git {REPO_DIR}
    print("클론 완료")
else:
    print("이미 클론됨")
%cd {REPO_DIR}


Cloning into '/content/StateDelta_Entropy'...
remote: Enumerating objects: 107, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 107 (delta 0), reused 1 (delta 0), pack-reused 102 (from 1)
Receiving objects: 100% (107/107), 45.37 MiB | 37.02 MiB/s, done.
Resolving deltas: 100% (32/32), done.
클론 완료
/content/StateDelta_Entropy


In [3]:
# ── STEP 3: 패키지 설치 ────────────────────────────
!pip install -q transformers==4.44.2 accelerate==1.2.1 termcolor pandas matplotlib 'numpy<2.0'
!pip install -q faiss-cpu==1.8.0
!pip install -q beir==1.0.1
!pip install -q elasticsearch==7.9.1 --no-deps
print("패키지 설치 완료")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 127.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 115.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requi

In [4]:
# ── STEP 4: HuggingFace 로그인 (필요시) ───────────
# Qwen은 공개 모델이라 토큰 없어도 됨
# from huggingface_hub import login
# login(token="hf_XXXXXXXXXX")


In [5]:
# ── STEP 5: root_path.py 설정 ──────────────────────
%cd /content/StateDelta_Entropy

with open("src/root_path.py", "w") as f:
    f.write('ROOT_DIR = "/content/StateDelta_Entropy"\n')
    f.write('ROOT_PATH = "/content/StateDelta_Entropy"\n')
    f.write('DATA_ROOT_PATH = "/content/StateDelta_Entropy/data"\n')

with open("src/root_path.py") as f:
    print(f.read())


/content/StateDelta_Entropy
ROOT_DIR = "/content/StateDelta_Entropy"
ROOT_PATH = "/content/StateDelta_Entropy"
DATA_ROOT_PATH = "/content/StateDelta_Entropy/data"



In [6]:
# ── STEP 6: utils.py 패치 ──────────────────────────
# generation_config.json 로컬 파일 접근 → HF hub에서 로드
with open("src/utils.py", "r") as f:
    code = f.read()

code = code.replace(
    '''def load_only_generation_config(model_name_or_path):
    config_path = os.path.join(model_name_or_path, "generation_config.json")
    assert os.path.exists(config_path), f"generation config {config_path} not found."
    generation_config = json.load(open(config_path, "r"))
    return generation_config''',
    '''def load_only_generation_config(model_name_or_path):
    from transformers import GenerationConfig
    gen_config = GenerationConfig.from_pretrained(model_name_or_path)
    return json.loads(gen_config.to_json_string())'''
)

with open("src/utils.py", "w") as f:
    f.write(code)
print("utils.py 패치 완료")


utils.py 패치 완료


In [7]:
# ── STEP 7: tasks/__init__.py 패치 (lazy import) ──
new_tasks = '''from src.tasks.ia import SingleIATask, NlIATask, CipherIATask, SDEIATask
from src.tasks.debate import SingleDebateTask, NlDebateTask, CipherDebateTask, SDEDebateTask
from src.tasks.utils import update_hidden_dim

def method_to_task_cls(task_type, method):
    if task_type == "workflow":
        from src.tasks.workflow import SingleWorkflowTask, NlWorkflowTask, CipherWorkflowTask, SDEWorkflowTask
        task_type_to_cls = {
            "workflow": {
                "single": SingleWorkflowTask,
                "nl": NlWorkflowTask,
                "cipher": CipherWorkflowTask,
                "sde": SDEWorkflowTask,
            }
        }
    else:
        task_type_to_cls = {
            "debate": {
                "single": SingleDebateTask,
                "nl": NlDebateTask,
                "cipher": CipherDebateTask,
                "sde": SDEDebateTask,
            },
            "ia": {
                "single": SingleIATask,
                "nl": NlIATask,
                "cipher": CipherIATask,
                "sde": SDEIATask,
            },
        }
    if task_type not in task_type_to_cls:
        raise NotImplementedError(f"Task type {task_type} not implemented.")
    if method not in task_type_to_cls[task_type]:
        raise NotImplementedError(f"Method {method} not implemented.")
    return task_type_to_cls[task_type][method]
'''

with open("src/tasks/__init__.py", "w") as f:
    f.write(new_tasks)
print("tasks/__init__.py 패치 완료")


tasks/__init__.py 패치 완료


In [8]:
# ── STEP 8: retriever/__init__.py 패치 (Qwen tokenizer) ──
with open("src/retriever/__init__.py", "r") as f:
    code = f.read()

code = code.replace(
    'AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")',
    'AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")'
)

with open("src/retriever/__init__.py", "w") as f:
    f.write(code)
print("retriever/__init__.py 패치 완료")


retriever/__init__.py 패치 완료


In [9]:
# ── STEP 9: sde_agent.py 교체 (엔트로피 계산 추가) ──
sde_agent_code = '''
import torch
import math
from typing import List, Optional, Callable, Union, Dict
from transformers import AutoTokenizer, GenerationConfig

from src.models.agents.base_agent import BaseAgent
from src.models.hs_edit_models import HiddenStatesEditingMethod, EditQwen2ForCausalLM, EditLlamaForCausalLM


class SDEAgent(BaseAgent):
    def __init__(
        self,
        engine_model_name_or_path: str,
        engine_tokenizer: AutoTokenizer,
        engine_model: Union[EditQwen2ForCausalLM, EditLlamaForCausalLM],
        generation_configs: dict,
        max_new_tokens: int,
        role_prompt: str = None,
    ):
        self.model_name = engine_model_name_or_path
        self.tokenizer = engine_tokenizer
        self.model = engine_model
        self.generation_configs = generation_configs
        self.max_new_tokens = max_new_tokens
        self.role_prompt = role_prompt
        self.agent_type = \'sde\'
        self.init_chat_template()

    def init_history(self, first_user_prompt: str):
        messages = []
        self.assistant_ids = []
        self.assistant_hs = []
        if self.role_prompt is not None:
            messages.append({"role": "system", "content": self.role_prompt})
        messages.append({"role": "user", "content": first_user_prompt})
        input_ids = self.tokenizer.apply_chat_template(messages, add_generation_prompt=True)
        self.history_ids = input_ids

    def generate(
        self,
        input_ids: List[int],
        if_edit: bool = False,
        edit_layer_idx: Optional[List[int]] = None,
        edit_mask: Optional[torch.BoolTensor] = None,
        edit_tensor: Optional[Dict[int, torch.FloatTensor]] = None,
    ):
        input_len = len(input_ids)
        input_ids_tensor = torch.tensor(input_ids).unsqueeze(0).to(self.model.device)
        attention_mask = torch.ones(input_ids_tensor.shape).to(self.model.device)

        edit_method = HiddenStatesEditingMethod(
            if_edit=if_edit,
            edit_layer_idx=edit_layer_idx,
            edit_mask=edit_mask,
            edit_tensor=edit_tensor,
        )

        with torch.no_grad():
            output = self.model.generate(
                input_ids=input_ids_tensor,
                attention_mask=attention_mask,
                max_new_tokens=self.max_new_tokens + 1,
                pad_token_id=self.tokenizer.pad_token_id if self.tokenizer.pad_token_id else self.tokenizer.eos_token_id,
                return_dict_in_generate=True,
                output_hidden_states=True,
                output_scores=True,
                edit_method=edit_method,
                **self.generation_configs,
            )

        output_ids = output.sequences[0][input_len:-1].tolist()

        # 엔트로피 계산
        entropy = self.compute_entropy(output.scores, top_k=20)

        output_hs = {}
        for layer_idx in edit_layer_idx:
            output_hs[layer_idx] = []
            saved = edit_method.saved_hs[layer_idx]
            assert len(saved) == len(output_ids) + 1
            for i in range(1, len(saved)):
                output_hs[layer_idx].append(saved[i] - saved[i - 1])
            if len(output_hs[layer_idx]) > 0:
                output_hs[layer_idx] = torch.stack(output_hs[layer_idx], dim=1)
            else:
                output_hs[layer_idx] = None

        self.assistant_ids.append(output_ids)
        self.assistant_hs.append(output_hs)
        self.history_ids = self.history_ids + output_ids + self.assistant_prompt_ed

        return output_ids, output_hs, entropy

    def compute_entropy(self, scores, top_k=20):
        """
        논문(Think Just Enough) 수식 그대로 구현
        scores: tuple of Tensor (1, vocab_size)
        반환: 정규화된 평균 엔트로피 (0~1)
        """
        if len(scores) == 0:
            return 0.0
        entropies = []
        for token_scores in scores:
            logits = token_scores[0]
            topk_logits, _ = torch.topk(logits, k=min(top_k, logits.shape[-1]))
            probs = torch.softmax(topk_logits, dim=-1)
            entropy = -torch.sum(probs * torch.log2(probs + 1e-9)).item()
            entropies.append(entropy)
        mean_entropy = sum(entropies) / len(entropies)
        max_entropy = math.log2(top_k)
        return mean_entropy / max_entropy

    def get_human_output(self, input_ids: List[int]):
        return self.tokenizer.decode(input_ids)
'''

with open("src/models/agents/sde_agent.py", "w") as f:
    f.write(sde_agent_code)
print("sde_agent.py 교체 완료")


sde_agent.py 교체 완료


In [10]:
# ── STEP 10: ia.py 로드 확인 ────────────────────────
ia_code = open("src/tasks/ia.py").read()
print("현재 ia.py 로드됨, 아래 셀에서 교체")


현재 ia.py 로드됨, 아래 셀에서 교체


In [11]:
# ── STEP 10-2: ia.py 전체 교체 ────────────────────
from google.colab import files
print("ia.py 파일을 업로드하세요")
uploaded = files.upload()  # ia.py 업로드

import shutil
shutil.copy("ia.py", "src/tasks/ia.py")
print("ia.py 교체 완료")


ia.py 파일을 업로드하세요


Saving ia.py to ia.py
ia.py 교체 완료


Quasar-T 데이터 셋 없어서 직접 다운

In [14]:
import os
os.makedirs("data/quasar_t", exist_ok=True)

# Quasar-T 공식 GitHub에서 직접 다운로드
!wget -O data/quasar_t/dev.json \
    https://raw.githubusercontent.com/bdhingra/quasar/master/dataset/quasar-t/questions/dev.json.gz

!gunzip data/quasar_t/dev.json.gz
print("Quasar-T 다운로드 완료")

--2026-03-22 14:10:24--  https://raw.githubusercontent.com/bdhingra/quasar/master/dataset/quasar-t/questions/dev.json.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-03-22 14:10:24 ERROR 404: Not Found.

gzip: data/quasar_t/dev.json.gz: No such file or directory
Quasar-T 다운로드 완료


In [15]:
!grep -r "quasar" /content/StateDelta_Entropy/src --include="*.py" -l
!grep -r "quasar" /content/StateDelta_Entropy/configs -l

/content/StateDelta_Entropy/src/main.py


In [16]:
!grep -n "quasar" /content/StateDelta_Entropy/src/main.py

45:    elif args.dataset == "quasart":


In [17]:
!cat /content/StateDelta_Entropy/src/main.py

import os
import sys
import json
import copy
import time
import torch
import hashlib
import argparse
from tqdm import tqdm
from termcolor import colored

from src.data import (
    GSM8K, MMLU, 
    WikiMultihopQA, StrategyQA, ComplexWebQuestions, QuasarT,
    HotpotQA, FEVER
)
from src.models.agents import NlAgent, CipherAgent, SDEAgent
from src.root_path import ROOT_PATH, DATA_ROOT_PATH
from src.utils import load_model, load_only_generation_config
from src.tasks import method_to_task_cls, update_hidden_dim


def generate_args_hash(
    setting: dict, 
    ignore_keys=["config_file", "sample", "redo", "sde_layer_config", "output_dir"],
):
    setting = {k: v for k, v in setting.items() if k not in ignore_keys}
    setting = json.dumps(setting, sort_keys=True)
    hash_output = hashlib.md5(setting.encode()).hexdigest()
    return hash_output


def main(args):
    if args.dataset == "gsm8k":
        dataset = GSM8K(data_root_path=DATA_ROOT_PATH)
    elif args.dataset.startswith("mmlu_")

In [19]:
import os
os.makedirs("/content/StateDelta_Entropy/data/QuasarT", exist_ok=True)

# Quasar-T 공식 GitHub에서 다운로드
!wget -O /content/StateDelta_Entropy/data/QuasarT/dev_questions.json.gz \
    https://raw.githubusercontent.com/bdhingra/quasar/master/dataset/quasar-t/questions/dev.json.gz

!gunzip /content/StateDelta_Entropy/data/QuasarT/dev_questions.json.gz
print("완료")

--2026-03-22 14:24:13--  https://raw.githubusercontent.com/bdhingra/quasar/master/dataset/quasar-t/questions/dev.json.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-03-22 14:24:13 ERROR 404: Not Found.


gzip: /content/StateDelta_Entropy/data/QuasarT/dev_questions.json.gz: unexpected end of file
완료


In [20]:
# ── STEP 11: Quasar-T 데이터셋 다운로드 ──────────────
import os
os.makedirs("data/quasar_t", exist_ok=True)

# Quasar-T dev 파일 경로 (StateDelta 레포 기준)
# 파일명: quasar_t_dev.json (또는 test.json — 레포 data/ 폴더 구조에 따라 확인)
dst = "data/quasar_t/dev.json"
if not os.path.exists(dst):
    print("Quasar-T 데이터셋을 수동으로 업로드하세요 (dev.json)")
    from google.colab import files
    uploaded = files.upload()  # dev.json 업로드
    import shutil
    # 업로드된 파일명에 맞게 조정
    uploaded_name = list(uploaded.keys())[0]
    shutil.copy(uploaded_name, dst)
    print("Quasar-T 완료")
else:
    print("Quasar-T 이미 존재")

# 레포 내 data/ 폴더에 이미 있을 경우 확인
import glob
existing = glob.glob("data/**/*quasar*", recursive=True) + glob.glob("data/**/*quasar_t*", recursive=True)
print("발견된 Quasar-T 관련 파일:", existing)


Quasar-T 이미 존재
발견된 Quasar-T 관련 파일: ['data/quasar_t', 'data/quasar_t']


In [21]:
# ── STEP 12: wikidpr_retrieval 압축 해제 ──────────
import os
if not os.path.exists("wikidpr_retrieval"):
    if os.path.exists("wikidpr_retrieval.tar.gz"):
        !tar -xzf wikidpr_retrieval.tar.gz
        print("wikidpr_retrieval 완료")
    else:
        print("wikidpr_retrieval.tar.gz 없음 - 업로드 필요")
        from google.colab import files
        uploaded = files.upload()  # wikidpr_retrieval.tar.gz 업로드
        !tar -xzf wikidpr_retrieval.tar.gz
        print("wikidpr_retrieval 완료")
else:
    print("wikidpr_retrieval 이미 존재")


wikidpr_retrieval 완료


In [22]:
# ── STEP 13: GPU 확인 ──────────────────────────────
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch: 2.10.0+cu128
CUDA 사용 가능: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 79.3 GB


quasar_t 데이터 비어잇는 문젷ㅎㅎ하

In [24]:
# 파일 크기 확인
!ls -lh /content/StateDelta_Entropy/data/QuasarT/
!ls -lh /content/StateDelta_Entropy/data/quasar_t/

total 0
-rw-r--r-- 1 root root 0 Mar 22 14:24 dev_questions.json.gz
total 0
-rw-r--r-- 1 root root 0 Mar 22 14:10 dev.json


In [25]:
import os
os.makedirs("/content/StateDelta_Entropy/data/QuasarT", exist_ok=True)

!wget "https://github.com/bdhingra/quasar/raw/master/dataset/quasar-t/questions/dev.json.gz" \
    -O /content/StateDelta_Entropy/data/QuasarT/dev_questions.json.gz

!gunzip -f /content/StateDelta_Entropy/data/QuasarT/dev_questions.json.gz

# 정상 확인
!wc -l /content/StateDelta_Entropy/data/QuasarT/dev_questions.json
!head -c 200 /content/StateDelta_Entropy/data/QuasarT/dev_questions.json

--2026-03-22 14:31:11--  https://github.com/bdhingra/quasar/raw/master/dataset/quasar-t/questions/dev.json.gz
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-03-22 14:31:11 ERROR 404: Not Found.


gzip: /content/StateDelta_Entropy/data/QuasarT/dev_questions.json.gz: unexpected end of file
wc: /content/StateDelta_Entropy/data/QuasarT/dev_questions.json: No such file or directory
head: cannot open '/content/StateDelta_Entropy/data/QuasarT/dev_questions.json' for reading: No such file or directory


In [26]:
!sed -n '385,420p' /content/StateDelta_Entropy/src/data.py


    def evaluate(self, output, test_id):
        return multihopqa_reevaluate(output, test_id, self.direct_evaluate)


class QuasarT(BaseDataset):
    stop_words = ["\\boxed{", "\\text{"]
    eval_metrics = ["em", "f1", "precision", "recall"]

    def __init__(self, data_root_path: str, retrieval_topk: int = 0):
        with open(os.path.join(data_root_path, "QuasarT/dev_questions.json"), "r") as fin:
            lines = fin.readlines()
        if retrieval_topk != 0:
            with open(os.path.join(ROOT_PATH, "wikidpr_retrieval", "QuasarT.json"), "r") as fin:
                retrieval_passages = json.load(fin)
        else:
            from src.retriever import bm25_retrieve
            retrieval_passages = None
        self.dataset = []
        for did, li in enumerate(lines):
            data = json.loads(li)
            val = {
                "qid": data["uid"], 
                "test_id": did, 
                "question": data["question"], 
                "answer": data["ans

quasar 다운 받는 방법 다시 readme에서 읽어옴

In [27]:
!cat /content/StateDelta_Entropy/README.md

# StateDeltaEncoding

Welcome to the Official Repository of State Delta Encoding (SDE)!

This repository contains the code, datasets models used in our paper: "Augmenting Multi-agent Communication with State Delta Trajectory".

![](figs/SDE_only.png)

## Reproduce Paper Results

### Install Environment

```bash
conda create -n SDE python=3.10.16
conda activate SDE
pip install -r requirements.txt
```

+ Update `ROOT_DIR` in `src/root_path.py` to to the folder address where you store this project.

+ Update `DATA_ROOT_PATH` in `src/root_path.py` to the dataset directory (e.g., `data/` in the current directory).

### Download Datasets

You can download the datasets used in our experiments as follows, assuming `DATA_ROOT_PATH` is `data/` in the current directory.

<details>

<summary>Quasar-T</summary>

Download the file `dev_questions.json` from the dataset link <http://curtis.ml.cmu.edu/datasets/quasar/quasar-t/questions/> and unzip it to `data/QuasarT/dev_questions.json`.

</details>

<

In [29]:
from google.colab import files
import shutil, os

os.makedirs("/content/StateDelta_Entropy/data/QuasarT", exist_ok=True)

print("dev_questions.json 파일을 업로드하세요")
uploaded = files.upload()

uploaded_name = list(uploaded.keys())[0]
shutil.copy(uploaded_name, "/content/StateDelta_Entropy/data/QuasarT/dev_questions.json")

# 정상 확인
!wc -l /content/StateDelta_Entropy/data/QuasarT/dev_questions.json
!head -c 300 /content/StateDelta_Entropy/data/QuasarT/dev_questions.json

dev_questions.json 파일을 업로드하세요


Saving dev_questions (1).json.gz to dev_questions (1).json.gz
517 /content/StateDelta_Entropy/data/QuasarT/dev_questions.json
�[�aY�dev_questions.json ��k��8�-�}~�����#��R��~V9m_��<}�'�	I�H��)�7��^�2����O��e;����Z��2/w������d^��x*��g��J������or't)dn���"��+c�n��Ht�d����N�;���h:�_*�.���D�����|�>�Yj�u�ʍ��?��������V�*�m/���ZӪ����/*-7Be������y4��+j}}?���R�}�[eM��?z��L�L.�+K��(k�
�Z��*Ň2�V�\D

In [30]:
import shutil, os

os.makedirs("/content/StateDelta_Entropy/data/QuasarT", exist_ok=True)

# 업로드된 파일 압축 해제
!gunzip -c "/content/dev_questions (1).json.gz" > /content/StateDelta_Entropy/data/QuasarT/dev_questions.json

# 정상 확인
!wc -l /content/StateDelta_Entropy/data/QuasarT/dev_questions.json
!head -c 300 /content/StateDelta_Entropy/data/QuasarT/dev_questions.json

gzip: /content/dev_questions (1).json.gz: No such file or directory
0 /content/StateDelta_Entropy/data/QuasarT/dev_questions.json


In [31]:
!find /content -name "*.gz" 2>/dev/null
!find /content -name "*.json.gz" 2>/dev/null
!find /content -name "*dev*" 2>/dev/null

/content/StateDelta_Entropy/data/QuasarT/dev_questions.json.gz
/content/StateDelta_Entropy/dev_questions (1).json.gz
/content/StateDelta_Entropy/wikidpr_retrieval.tar.gz
/content/StateDelta_Entropy/data/QuasarT/dev_questions.json.gz
/content/StateDelta_Entropy/dev_questions (1).json.gz
/content/StateDelta_Entropy/data/QuasarT/dev_questions.json
/content/StateDelta_Entropy/data/QuasarT/dev_questions.json.gz
/content/StateDelta_Entropy/data/quasar_t/dev.json
/content/StateDelta_Entropy/dev_questions (1).json.gz


In [34]:
!gunzip -c "/content/StateDelta_Entropy/dev_questions (1).json.gz" > /content/StateDelta_Entropy/data/QuasarT/dev_questions.json

# 확인
!wc -l /content/StateDelta_Entropy/data/QuasarT/dev_questions.json
!head -c 300 /content/StateDelta_Entropy/data/QuasarT/dev_questions.json

3000 /content/StateDelta_Entropy/data/QuasarT/dev_questions.json
{"answer": "tetanus", "question": "Lockjaw is another name for which disease", "uid": "s3q1674", "tags": ["1tok", "yes-answer-long", "yes-answer-short"]}
{"answer": "leek", "question": "Which vegetable is a Welsh emblem ?", "uid": "s3q18157", "tags": ["1tok"]}
{"answer": "the guns of naverone", "que

로드 실패한 quasar-t 파일들 삭제

In [36]:
import os

os.remove("/content/StateDelta_Entropy/data/QuasarT/dev_questions.json.gz")
os.remove("/content/StateDelta_Entropy/data/quasar_t/dev.json")
os.remove("/content/StateDelta_Entropy/dev_questions (1).json.gz")

# quasar_t 빈 폴더도 삭제
os.rmdir("/content/StateDelta_Entropy/data/quasar_t")

print("삭제 완료")

삭제 완료


In [35]:
# ── STEP 14: 실험 실행 (sample=1 테스트) ──────────
%cd /content/StateDelta_Entropy

!python -m src.main \
    --config_file configs/ia.json \
    --model_name_or_path Qwen/Qwen2.5-7B-Instruct \
    --dataset quasart \
    --method sde \
    --sample 1 \
    --redo


/content/StateDelta_Entropy
/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Namespace(config_file='configs/ia.json', model_name_or_path='Qwen/Qwen2.5-7B-Instruct', dataset='quasart', sample=1, method='sde', output_dir='output', redo=True, task_type='ia', rounds=5, agent_cnt=2, generation_setting='greedy', max_new_tokens=256, prompt_file='prompts/ia.yaml', retrieval_topk=6, edit_layer_idx=[22], sde_layer_config='configs/sde_layer.json')
### Output dir: /content/StateDelta_Entropy/output/ia/quasart/Qwen2.5-7B-Instruct/sde/1f86b68fe13893fa3b83cf8f77b48734/run_0 ###
config.json: 100% 663/663 [00:00<00:00, 4.84MB/s]
model.safetensors.index.json: 27.8kB [00:00, 81.4MB/s]
model-00001-of-00004.safetensors:   0% 0.00/3.95G [00:00<?, ?B/s]
model-00001-of-00004.safetensors:   0% 0.00/3.95G [00:00<?, ?B/s]
model-00001-of-00004.safetensors

In [38]:
# ── STEP 15: 실험 실행 (sample=300 본실험) ────────
# 테스트 완료 후 이 셀 실행
%cd /content/StateDelta_Entropy

!python -m src.main \
    --config_file configs/ia.json \
    --model_name_or_path Qwen/Qwen2.5-7B-Instruct \
    --dataset quasart \
    --method sde \
    --sample 300 \
    --redo


/content/StateDelta_Entropy
/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Namespace(config_file='configs/ia.json', model_name_or_path='Qwen/Qwen2.5-7B-Instruct', dataset='quasart', sample=300, method='sde', output_dir='output', redo=True, task_type='ia', rounds=5, agent_cnt=2, generation_setting='greedy', max_new_tokens=256, prompt_file='prompts/ia.yaml', retrieval_topk=6, edit_layer_idx=[22], sde_layer_config='configs/sde_layer.json')
### Output dir: /content/StateDelta_Entropy/output/ia/quasart/Qwen2.5-7B-Instruct/sde/1f86b68fe13893fa3b83cf8f77b48734/run_1 ###
Loading checkpoint shards: 100% 4/4 [00:13<00:00,  3.32s/it]
  0% 0/300 [00:00<?, ?it/s]2026-03-22 14:52:00.571582: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off e

In [39]:
# ── STEP 16: 결과 출력 ────────────────────────────
import glob, json, os
from collections import defaultdict

ROOT = "/content/StateDelta_Entropy"
MODEL_NAME = "Qwen2.5-7B-Instruct"

# 논문 참고값 (Quasar-T / Qwen2.5-7B-Instruct)
PAPER = {
    "sde":    {"em": 0.3150, "f1": 0.3772},
    "nl":     {"em": 0.3050, "f1": 0.3748},
    "cipher": {"em": 0.2817, "f1": 0.3567},
}

def load_latest(file_type):
    pattern = f"{ROOT}/output/ia/quasart/{MODEL_NAME}/sde/**/{file_type}.json"
    files = sorted(glob.glob(pattern, recursive=True), key=os.path.getmtime)
    if not files:
        return None
    with open(files[-1]) as f:
        return json.load(f)

r = load_latest("result")
detail = load_latest("detail")

if r is None:
    print("결과 없음")
else:
    avg = r["average"]
    sep = "=" * 60

    # 논문 재현 검증
    print(f"\n{sep}")
    print(f"  📊 논문 재현 검증 — Quasar-T / SDE+Entropy")
    print(sep)
    print(f"  {'지표':<12} {'논문':>8} {'재현':>8} {'차이':>8}")
    print(f"  {'-'*40}")
    for metric in ["em", "f1", "precision", "recall"]:
        if metric not in avg:
            continue
        repro = avg[metric]
        paper = PAPER["sde"].get(metric)
        paper_str = f"{paper:.4f}" if paper else "—"
        diff_str  = f"{repro-paper:+.4f}" if paper else "—"
        print(f"  {metric.upper():<12} {paper_str:>8} {repro:>8.4f} {diff_str:>8}")

    # 분석
    print(f"\n{sep}")
    print(f"  🔍 분석")
    print(sep)
    print(f"  샘플 수:         {r.get('sample')}개")
    print(f"  평균 추론시간:   {r.get('run_time(s)', 0):.1f}초")
    print(f"  사용 라운드:     {r.get('use_rounds', '—')} / {r.get('rounds', '—')}")

    # 엔트로피 통계
    if detail:
        all_logs = [log for d in detail for log in d.get("entropy_log", [])]
        if all_logs:
            sde_cnt = sum(1 for l in all_logs if l["method"] == "SDE")
            nl_cnt  = sum(1 for l in all_logs if l["method"] == "NL")
            all_entropy_vals = [e for l in all_logs for e in (l["entropy"] if isinstance(l["entropy"], list) else [l["entropy"]])]
            avg_ent = sum(all_entropy_vals) / len(all_entropy_vals)
            total   = len(all_logs)

            print(f"\n{sep}")
            print(f"  📈 엔트로피 통계 (임계값=0.2)")
            print(sep)
            print(f"  총 통신 횟수:    {total}회")
            print(f"  SDE 선택:        {sde_cnt}회 ({sde_cnt/total*100:.1f}%)")
            print(f"  NL  선택:        {nl_cnt}회  ({nl_cnt/total*100:.1f}%)")
            print(f"  평균 엔트로피:   {avg_ent:.4f}")

            rd_entropy = defaultdict(list)
            for log in all_logs:
                for e in (log["entropy"] if isinstance(log["entropy"], list) else [log["entropy"]]):
                  rd_entropy[log["round"]].append(e)
            print(f"\n  라운드별 평균 엔트로피:")
            for rd in sorted(rd_entropy.keys()):
                vals = rd_entropy[rd]
                print(f"    round {rd}: avg={sum(vals)/len(vals):.4f}  "
                      f"min={min(vals):.4f}  max={max(vals):.4f}")
        else:
            print("  엔트로피 로그 없음")
    print(sep)



  📊 논문 재현 검증 — Quasar-T / SDE+Entropy
  지표                 논문       재현       차이
  ----------------------------------------
  EM             0.3150   0.3100  -0.0050
  F1             0.3772   0.3738  -0.0034
  PRECISION           —   0.3743        —
  RECALL              —   0.4262        —

  🔍 분석
  샘플 수:         300개
  평균 추론시간:   29.9초
  사용 라운드:     2.6666666666666665 / 5

  📈 엔트로피 통계 (임계값=0.2)
  총 통신 횟수:    1108회
  SDE 선택:        207회 (18.7%)
  NL  선택:        901회  (81.3%)
  평균 엔트로피:   0.1534

  라운드별 평균 엔트로피:
    round 1: avg=0.1594  min=0.0204  max=0.2923
    round 2: avg=0.1552  min=0.0251  max=0.3089
    round 3: avg=0.1391  min=0.0137  max=0.2791
    round 4: avg=0.0983  min=0.0174  max=0.2013


In [40]:
# ── STEP 17: 결과 구글 드라이브 저장 ──────────────
import shutil
from google.colab import drive
drive.mount('/content/drive')

shutil.copytree(
    "/content/StateDelta_Entropy/output",
    "/content/drive/MyDrive/StateDelta_Entropy/output",
    dirs_exist_ok=True
)
print("드라이브 저장 완료")


MessageError: Error: credential propagation was unsuccessful